In [18]:
IMAGE_PATH = r"M:\dev\rcmreloaded\sunset.webp"
GRID_STEPS = 32       # samples per RGB channel (8^3 = 512 points)
BANDWIDTH = 3
THRESHOLD = -15      # log-density floor — points below this are transparent
LEAF_SIZE = 1000
MAX_FIT_PIXELS = 20_000  # random subsample used to fit the KDE
SATURATION_PENALTY = 5    # subtracts up to this from log-density for fully unsaturated colours (0 = off)

In [13]:
import numpy as np
from PIL import Image
from skimage import color
from sklearn.neighbors import KernelDensity
import plotly.graph_objects as go

In [14]:
# Fit KDE on image pixels in LAB space
img = Image.open(IMAGE_PATH).convert("RGBA")
pixels_rgba = np.array(img).reshape(-1, 4)

# Drop transparent pixels before fitting
pixels_rgb = pixels_rgba[pixels_rgba[:, 3] > 0, :3] / 255.0
pixels_lab = color.rgb2lab(pixels_rgb.reshape(-1, 1, 3)).reshape(-1, 3)

if len(pixels_lab) > MAX_FIT_PIXELS:
    idx = np.random.choice(len(pixels_lab), MAX_FIT_PIXELS, replace=False)
    pixels_lab = pixels_lab[idx]

kde = KernelDensity(kernel="gaussian", bandwidth=BANDWIDTH, leaf_size=LEAF_SIZE)
kde.fit(pixels_lab)
print(f"KDE fitted on {len(pixels_lab)} pixels")

KDE fitted on 20000 pixels


In [15]:
# Sample a uniform grid across RGB space
steps = np.linspace(0, 255, GRID_STEPS, dtype=int)
r, g, b = np.meshgrid(steps, steps, steps)
grid_rgb = np.stack([r.ravel(), g.ravel(), b.ravel()], axis=1)  # (N, 3) uint8

# Convert grid to LAB and score
grid_rgb_norm = grid_rgb / 255.0

In [16]:
grid_lab = color.rgb2lab(grid_rgb_norm.reshape(-1, 1, 3)).reshape(-1, 3)

In [17]:
log_density = kde.score_samples(grid_lab)

# Penalise unsaturated (grey) colours
grid_hsv = color.rgb2hsv(grid_rgb_norm.reshape(-1, 1, 3)).reshape(-1, 3)
saturation = grid_hsv[:, 1]  # 0 = grey, 1 = fully saturated
log_density = log_density - GREY_PENALTY * (1 - saturation)

In [273]:
# Anything below THRESHOLD is transparent; above maps linearly to [0, 1]
hi = log_density.max()
alpha = np.clip((log_density - THRESHOLD) / (hi - THRESHOLD), 0, 1)
print(f"Grid size: {len(grid_rgb)}, density range: [{log_density.min():.2f}, {log_density.max():.2f}]")

Grid size: 32768, density range: [-832.35, -10.47]


In [ ]:
# Only render points above a meaningful density threshold
mask = alpha > 0.1
grid_rgb_vis = grid_rgb[mask]
alpha_vis = alpha[mask]

# Map alpha to marker size range [2, 12]
sizes = 0.2 + alpha_vis * 10

colours = [f"rgb({r},{g},{b})" for r, g, b in grid_rgb_vis]

fig = go.Figure(go.Scatter3d(
    x=grid_rgb_vis[:, 0],
    y=grid_rgb_vis[:, 1],
    z=grid_rgb_vis[:, 2],
    mode="markers",
    marker=dict(size=sizes, color=colours, line=dict(width=0)),
))

fig.update_layout(
    scene=dict(
        xaxis_title="R",
        yaxis_title="G",
        zaxis_title="B",
        xaxis=dict(range=[0, 255]),
        yaxis=dict(range=[0, 255]),
        zaxis=dict(range=[0, 255]),
    ),
    title="RGB colour space — size = KDE density",
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

In [26]:
rgb_normalised = np.array([0.01, 0.5, 0.95]).reshape(1, 3)

grid_lab = color.rgb2lab(rgb_normalised)
grid_hsv = color.rgb2hsv(rgb_normalised)  # we use HSV because it has a constant max/min saturation
saturation = grid_hsv[:, 1]  # 0 = grey, 1 = fully saturated


log_density = kde.score_samples(grid_lab)
log_density = log_density - SATURATION_PENALTY * (1 - saturation)

In [ ]:
# Anything below THRESHOLD is transparent; above maps linearly to [0, 1]
hi = log_density.max()
alpha = np.clip((log_density - THRESHOLD) / (hi - THRESHOLD), 0, 1)
print(f"Grid size: {len(grid_rgb)}, density range: [{log_density.min():.2f}, {log_density.max():.2f}]")